# RNN Sentiment Analysis — Beginner Notebook

We are going to teach a computer to read a movie review and decide:
**is it POSITIVE or NEGATIVE?**

We will use an **RNN (Recurrent Neural Network)** — a type of neural network
that reads text *one word at a time*, keeping a running "memory" as it goes.
This makes it good at understanding sentences, where **word order matters**.

We'll use real movie reviews from the IMDB dataset (already built into Keras),
and a library called **TensorFlow / Keras** to build and train the model.

Run each cell from top to bottom. Every code cell has a markdown cell above
it explaining, in simple words, what it does and *why*.

## Step 1 — Import the tools we need

- `imdb` gives us 50,000 real movie reviews, already labeled positive/negative.
- `pad_sequences` will help us make every review the same length later.
- `Sequential`, `Embedding`, `SimpleRNN`, `Dense` are the building blocks we'll
  use to construct our model.

In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

print("TensorFlow version:", tf.__version__) 

TensorFlow version: 2.21.0


## Step 2 — Load the movie review data

We only keep the **10,000 most common words** (`VOCAB_SIZE`). Rare words
(typos, unusual names) are ignored — they rarely help decide sentiment, and
keeping the vocabulary small keeps the model fast.

Each review is already converted into a list of numbers by Keras — every
number stands for one word (e.g. `1` = the most common word, `2` = the next
most common, and so on).

`y_train` / `y_test` are the labels: `1` = positive review, `0` = negative
review.

In [2]:
VOCAB_SIZE = 10000

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

print("Number of training reviews:", len(x_train))
print("Number of test reviews:", len(x_test))
print()
print("A review, as numbers (first 20 numbers only):")
print(x_train[0][:20])
print()
print("Its label (1 = positive, 0 = negative):", y_train[0])

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 30s 2us/step
Number of training reviews: 25000
Number of test reviews: 25000

A review, as numbers (first 20 numbers only):
[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25]

Its label (1 = positive, 0 = negative): 1


## Step 3 — Make every review the same length

Reviews are all different lengths, but a neural network needs a **fixed-size**
input every time. `pad_sequences` fixes this:

- Long reviews get **cut short** to `MAX_LEN` words.
- Short reviews get **padded with zeros** at the front, up to `MAX_LEN` words.

We're choosing 200 words — usually enough to capture the sentiment of a
review without wasting time on extra-long text.

In [3]:
MAX_LEN = 200

x_train = pad_sequences(x_train, maxlen=MAX_LEN)
x_test = pad_sequences(x_test, maxlen=MAX_LEN)

print("Shape of x_train after padding:", x_train.shape)
print("Shape of x_test after padding:", x_test.shape)

Shape of x_train after padding: (25000, 200)
Shape of x_test after padding: (25000, 200)


## Step 4 — Build the RNN model

We stack three layers, one after another (`Sequential`):

1. **`Embedding`** — turns each word-number into a list of 32 numbers (a
   "meaning vector"). Words with similar meaning end up with similar vectors.
   A raw word ID like `47` carries no meaning by itself, but a 32-number
   vector can — the model learns these vectors during training.

   input_dim=10000 means: "Build exactly 10,000 folders."

   output_dim=32 means: "Inside every folder, use exactly 32 different number scores to describe the personality of that word."

   input_length=MAX_LEN: This just tells the translator how many words are coming. It says, "Get ready, because every single movie review I hand you will be exactly 200 words long."

2. **`SimpleRNN(32)`** — this is the actual RNN. It reads the 200 word-vectors
   **one at a time, left to right**, and keeps a 32-number "memory" that gets
   updated at every word. This is what lets it understand things like
   "not good" being different from "good".

   # The Problem: Computers have no memory
      Normally, if you hand a computer a sentence like "The movie was great", it looks at all four words at the exact same time. It has no concept of order.

      But humans don't read that way. When you read a sentence, you read one word at a time, from left to right. As you read, you keep a "running memory" in your brain of what the sentence is about so far.

      A SimpleRNN is a special tool that forces the computer to read exactly like a human: one word at a time, from left to right, while holding onto a memory.

      What is units=32? (The size of the memory notepad)
      Imagine I give you a small notepad while you are reading a long book. I tell you: "Every time you read a word, write down some notes about the story so far."

      That is what units=32 is. It is the size of the computer's notepad.

      It tells the computer: "As you read this movie review, you are allowed to use exactly 32 numbers to write down your memory of what the review is about."

      How it works step-by-step:
      Let's say the computer reads a short review: "I hated this movie."

      Reads "I": The RNN looks at the word "I". It takes its notepad (the 32 numbers) and jots down a quick note. (Memory: Okay, someone is talking about themselves.)

      Reads "hated": The RNN looks at the word "hated." It looks at its previous memory note. It updates the notepad with new information. (Memory: Whoa, negative feeling! Someone is very angry about something!)

      Reads "this": It updates the notepad again.

      Reads "movie": It updates the notepad one last time. (Memory: The anger is directed at a movie. This is a very negative review.)

      When the RNN finishes the very last word of the sentence, it closes the notepad. Those final 32 numbers contain the full summary of the entire sentence!

      The RNN then takes that notepad and hands it directly to Machine #3 (the Dense layer) to make the final Positive/Negative decision.

      To summarize: units=32 just means the RNN uses a list of 32 numbers to keep track of its "running memory" as it reads the sentence word by word.

3. **`Dense(1, activation='sigmoid')`** — squashes the RNN's final memory
   down into **one number between 0 and 1**: our positive/negative score.
   Close to 1 = positive, close to 0 = negative.
   
   This is Machine #3 on the assembly line. Its job is to be the Final Judge.

   ## Easy Explain:

   Dense(1): "Dense" is just the coding name for a standard Artificial Neural Network (ANN) layer. The 1 tells it that we only want one single number as the final output. We don't want a list of numbers; we just want one final grade.

   activation='sigmoid': This is a special math tool called a "squisher." The RNN's memory might output a crazy number like 845 or -203. The "sigmoid" tool catches that crazy number and forces it to squish down so it fits perfectly between 0 and 1.

   If the math squishes to 0.90, the review is Positive.

   If the math squishes to 0.10, the review is Negative.

In [4]:
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=32, input_length=MAX_LEN),
    SimpleRNN(units=32),
    Dense(1, activation='sigmoid')
])

model.summary()

c:\Users\Primax\anaconda3\envs\venv\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Step 5 — Compile the model

This step doesn't train anything yet — it just tells Keras *how* training
should work:

- **`optimizer='adam'`** — the algorithm that decides how to adjust the
  model's internal numbers after each mistake. A reliable, common default.
- **`loss='binary_crossentropy'`** — how we measure "how wrong" a prediction
  was. This is the standard choice whenever the answer is one of two classes
  (positive / negative).
- **`metrics=['accuracy']`** — just so we can *see* accuracy (% correct)
  printed while training — it doesn't affect the actual learning.

Before an AI model can actually start reading data and learning, you have to set the "rules of the game."

The model.compile(...) step is where you hire the staff that will train your AI. You are choosing the teacher, the grading system, and the report card.

Here is what each specific piece does in simple English:

## 1. optimizer='adam' (The Smart Coach)
When the AI makes a mistake (like guessing a review is Positive when it was actually Negative), it needs someone to tell it how to fix its internal "volume knobs" (the weights) so it doesn't make that mistake again.

What is it? The optimizer is the coach.

Why 'adam'? "Adam" is currently the most popular and famous coach in deep learning. It is very smart because it automatically adjusts how fast or slow the AI learns. If the AI is completely lost, Adam makes it take big learning steps. If the AI is close to the right answer, Adam makes it take tiny, careful steps.

## 2. loss='binary_crossentropy' (The Mistake Calculator)
The AI needs a mathematical way to know exactly how wrong it is.

What is it? The loss function is a mathematical formula that calculates the AI's mistakes. The goal of the AI is to make this "loss" number as close to zero as possible.

Why 'binary'? "Binary" means there are only two possible choices. In our movie project, the answer is always either exactly 0 (Negative) or 1 (Positive).

Why 'crossentropy'? This is just the name of the specific math formula that is best at calculating distance between a 0 and a 1. (For example, if the AI guesses 0.8 but the real answer was 1.0, this formula calculates exactly how much of a penalty to give the AI).

## 3. metrics=['accuracy'] (The Human Report Card)
While the computer loves complex math formulas like "binary crossentropy," humans hate reading them. We just want a simple percentage.

What is it? This tells the computer to print out a normal, easy-to-read grade for you while it trains.

Why 'accuracy'? It simply means: "Out of 100 movie reviews, how many did you guess correctly?" So, as the AI trains, you will see a nice clean number like 0.85 on your screen, which just means the AI is getting an 85% on its test.

In [5]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

## Step 6 — Train the model

Now the model actually learns.

- **`epochs=5`** — the model studies the entire training set 5 full times.
- **`batch_size=128`** — it looks at 128 reviews at once, then makes one small
  update to itself, rather than updating after every single review.
- **`validation_split=0.2`** — 20% of the training reviews are set aside just
  to check progress after each epoch (not used for actual learning). This
  warns us early if the model starts memorizing instead of understanding.

This step may take a few minutes.

In [6]:
history = model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2
)

Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 9s 43ms/step - accuracy: 0.5791 - loss: 0.6689 - val_accuracy: 0.5996 - val_loss: 0.6917
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 6s 37ms/step - accuracy: 0.7735 - loss: 0.4859 - val_accuracy: 0.7764 - val_loss: 0.4829
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 6s 37ms/step - accuracy: 0.8974 - loss: 0.2591 - val_accuracy: 0.8024 - val_loss: 0.4580
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 6s 39ms/step - accuracy: 0.9672 - loss: 0.1088 - val_accuracy: 0.7814 - val_loss: 0.5531
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 6s 37ms/step - accuracy: 0.9940 - loss: 0.0365 - val_accuracy: 0.8036 - val_loss: 0.5902


## Step 7 — Evaluate on unseen test data

`x_test` / `y_test` are reviews the model has **never seen during training**.
This is the only honest way to check whether it actually learned to
understand sentiment, rather than just memorizing the training reviews.

In [7]:
loss, accuracy = model.evaluate(x_test, y_test)
print(f"\nTest Accuracy: {accuracy * 100:.2f}%")

782/782 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.8027 - loss: 0.5938

Test Accuracy: 80.27%


## Step 8 — Try it on our own sentences

To test a brand-new sentence, we must convert it into numbers **the exact
same way** the training data was converted. `imdb.get_word_index()` gives us
the same word → number dictionary the dataset itself was built with.

The helper function below:
- lower-cases and splits the sentence into words
- starts with `1` (the dataset's "start of review" marker)
- looks up each word's number (unknown words become `2`)
- adds `3` to every number, because `0`, `1`, `2` are reserved for special
  meanings (padding, start, unknown) in this dataset
- caps any number above our vocabulary limit back down to "unknown"

Without this exact matching, the model would see garbage instead of real
words.

In [8]:
word_index = imdb.get_word_index()

def encode_review(text):
    words = text.lower().split()
    encoded = [1]  # 1 = "start of review" marker
    for word in words:
        idx = word_index.get(word, 2) + 3
        encoded.append(idx if idx < VOCAB_SIZE else 2)
    return encoded

def predict_sentiment(text):
    encoded = pad_sequences([encode_review(text)], maxlen=MAX_LEN)
    score = model.predict(encoded, verbose=0)[0][0]
    label = "Positive 🙂" if score > 0.5 else "Negative 🙁"
    print(f"Review: \"{text}\"")
    print(f"Prediction: {label}  (score: {score:.2f})")
    print()

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 2s 1us/step


## Step 9 — Test with a POSITIVE example

Let's try a sentence that a human would clearly call positive, and see if
the model agrees.

In [12]:
predict_sentiment("This movie was average and not very interesting.")

Review: "This movie was average and not very interesting."
Prediction: Positive 🙂  (score: 0.63)



## Step 10 — Test with a NEGATIVE example

Now let's try a sentence that a human would clearly call negative.

In [13]:
predict_sentiment("this movie was good but not great. I liked the acting but the plot was predictable.")

Review: "this movie was good but not great. I liked the acting but the plot was predictable."
Prediction: Negative 🙁  (score: 0.02)



### A note on these predictions

You might notice the model doesn't always get short, custom sentences right —
even ones that seem obviously positive or negative to us. This is expected,
and it's a useful lesson in itself:

- The model was trained on **full-length movie reviews** (up to 200 words),
  not short one-line sentences. A 7-word sentence is quite different from
  what it learned on.
- Training accuracy reached ~92% but test accuracy was only ~82% — the gap
  means the model **overfit** a little (memorized some training patterns
  rather than fully general rules).
- We only trained for 5 epochs with a small, simple `SimpleRNN`. More
  epochs, more units, or switching to `LSTM`/`GRU` (see the Recap) usually
  improves this.

None of this means the code is broken — it means this small model has real,
expected limits, just like the ones described earlier in the lesson.

## Step 11 — Try your own sentence

Change the text below to anything you like, and re-run the cell.

In [17]:
predict_sentiment("the hero was not good but the villain was great.")

Review: "the hero was not good but the villain was great."
Prediction: Negative 🙁  (score: 0.09)



## Recap — what we just built

1. **Loaded data** — 50,000 real movie reviews, already labeled.
2. **Padded sequences** — made every review exactly 200 words long.
3. **Built a model** — `Embedding` (word → meaning) → `SimpleRNN` (reads in
   order, keeps memory) → `Dense` (turns memory into a 0–1 score).
4. **Compiled** — chose how the model learns and how mistakes are measured.
5. **Trained** — showed it thousands of labeled reviews, 5 times over.
6. **Evaluated** — checked accuracy on reviews it had never seen.
7. **Predicted** — fed in our own sentences and got Positive/Negative back.

**Good to know:** a plain `SimpleRNN` has a *short memory* — it can start
forgetting things from many words ago in a long review. Swapping
`SimpleRNN(32)` for `LSTM(32)` or `GRU(32)` (also in
`tensorflow.keras.layers`) is often a one-line change that gives the model
a much longer memory.